---

---

# Environment Setup and Cuda Check

---


In [3]:
import os

# Add the directory containing the executable to the PATH
os.environ["PATH"] += os.pathsep + "/usr/local/cuda/bin"

# Check if the directory is added to the PATH
print(os.environ["PATH"])

/opt/tljh/user/bin:/bin:/usr/bin:/usr/local/cuda/bin


In [4]:
%%bash
nvcc --version
nvprof --version
nsys --version
ncu --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Apr__9_19:24:57_PDT_2025
Cuda compilation tools, release 12.9, V12.9.41
Build cuda_12.9.r12.9/compiler.35813241_0
nvprof: NVIDIA (R) Cuda command line profiler
Copyright (c) 2012 - 2025 NVIDIA Corporation
Release version 12.9.19 (21)
NVIDIA Nsight Systems version 2025.1.3.140-251335620677v0
NVIDIA (R) Nsight Compute Command Line Profiler
Copyright (c) 2018-2025 NVIDIA Corporation
Version 2025.2.0.0 (build 35613519) (public-release)


In [5]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Wed Nov  5 09:43:04 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.51.03              Driver Version: 575.51.03      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100-PCIE-32GB           Off |   00000000:00:10.0 Off |                    0 |
| N/A   26C    P0             22W /  250W |       0MiB /  32768MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

---
---
# Data Initialization Kernel
---

In [4]:
%%writefile DataInit.cu
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <math.h>
#include <curand_kernel.h>

__global__ void initializeData( float *A, float *X, int m, int n)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = gridDim.x * blockDim.x;

    for (int i = idx; i < n; i += stride) {
        X[i] = sinf(i * 0.01f) * cosf(i * 0.007f) + 0.01f;
    }

    for (int i = idx; i < m * n; i += stride) {
        int row = i / n; // Calculate row
        int col = i % n; // Calculate col
        A[i] = 1.0f / ((row + 1.0f) * (col + 1.0f));
    }
}

Writing DataInit.cu


---
---
# Variant 1: C/C++

In [5]:
%%writefile C_matvec_var1.c

#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <math.h>

// *** C function version: y = A * x (row-major A, single precision)
void matvec(size_t m, size_t n, float *A, float *x, float *y)
{
    for (size_t i = 0; i < m; i++) {
        float sum = 0.0f;
        for (size_t j = 0; j < n; j++) {
            sum += A[i * n + j] * x[j];
        }
        y[i] = sum;
    }
}

int main(void)
{
    const size_t N = 4096;          // m = n = 256
    const size_t M = N;
    const size_t A_BYTES = M * N * sizeof(float);
    const size_t X_BYTES = N * sizeof(float);
    const size_t Y_BYTES = M * sizeof(float);
    const size_t loope = 30;       // for averaging

    // declare arrays
    float *A = (float*)malloc(A_BYTES);
    float *x = (float*)malloc(X_BYTES);
    float *y = (float*)malloc(Y_BYTES);

    clock_t start, end;

    // init
     for (size_t i = 0; i < M; i++){
        for (size_t j = 0; j < N; j++){
            A[i*N+ j] = sinf(i * 0.002f + j * 0.001f);
        }
    }
    for (size_t i = 0; i < N; i++)
        x[i] = cosf(i * 0.003f);

    // fill-in cache (warm-up)
    matvec(M, N, A, x, y);

    // time here
    double elapse = 0.0, time_taken = 0.0;
    for (size_t i = 0; i < loope; i++) {
        start = clock();
        matvec(M, N, A, x, y);
        end = clock();
        time_taken = ((double)(end - start)) * 1E3 / CLOCKS_PER_SEC;
        elapse += time_taken;
    }

    printf("Function (in C) average time for %lu loops is %f milliseconds to execute a matrix-vector with size %lux%lu \n",
           loope, elapse / loope, M, N);

    // show results: first 3 + last 3
    printf("y[0..2]   = %f %f %f\n", y[0], y[1], y[2]);
    printf("y[-3..-1] = %f %f %f\n", y[M-3], y[M-2], y[M-1]);

    // error checking
    size_t err_count = 0;
    for (size_t i = 0; i < M; i++) {
        float sum = 0.0f;
        for (size_t j = 0; j < N; j++) {
            sum += A[i * N + j] * x[j];
        }
        if (y[i] != sum)
            err_count++;
    }
    printf("Error count (C program): %lu\n", err_count);

    // Free memory
    free(A);
    free(x);
    free(y);
    return 0;
}


Writing C_matvec_var1.c


In [6]:
%%bash
gcc -O3 -Wall -Wextra C_matvec_var1.c -o C_matvec_var1 -lm

In [7]:
%%bash
./C_matvec_var1

Function (in C) average time for 30 loops is 77.856500 milliseconds to execute a matrix-vector with size 4096x4096 
y[0..2]   = -110.005173 -109.688629 -109.370766
y[-3..-1] = 185.623276 185.726257 185.830643
Error count (C program): 0


---
---
# Variant 2: CUDA program version using a grid-stride loop without prefetch and without mem advise
---

In [8]:
%%writefile CUDA_MATVEC_VAR2.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

// CUDA MATVEC kernel (grid-stride loop)
__global__
void matvec(size_t m, size_t n, const float *A, const float *x, float *y){
    int row  = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    for (int i = row; i < (int)m; i += stride){
        float sum = 0.0f;
        for (size_t j = 0; j < n; j++){
            sum += A[i*n + j] * x[j];
        }
        y[i] = sum;
    }
}

int main() {
    const size_t m = 4096;
    const size_t n = 4096;
    const size_t MATRIX_SIZE = m * n;
    const size_t MATRIX_BYTES = MATRIX_SIZE * sizeof(float);
    const size_t VECTOR_BYTES = n * sizeof(float);
    const size_t LOOPS = 30;

    float *A, *x, *y;
    cudaMallocManaged(&A, MATRIX_BYTES);
    cudaMallocManaged(&x, VECTOR_BYTES);
    cudaMallocManaged(&y, VECTOR_BYTES);

    // Initialize A and x
    for (size_t i = 0; i < m; i++){
        for (size_t j = 0; j < n; j++){
            A[i*n + j] = sinf(i * 0.002f + j * 0.001f);
        }
    }
    for (size_t i = 0; i < n; i++)
        x[i] = cosf(i * 0.003f);

    // CUDA kernel launch setup
    size_t numThreads = 1024;
    size_t numBlocks = (m + numThreads - 1) / numThreads;

    printf("*** function = MATVEC (float)\n");
    printf("m = %lu, n = %lu (A elements = %lu)\n", m, n, MATRIX_SIZE);
    printf("numBlocks = %lu, numThreads = %lu\n",
           (unsigned long)numBlocks, (unsigned long)numThreads);

    // Multiple runs for nvprof timing
    for (size_t i = 0; i < LOOPS; i++)
        matvec<<<numBlocks, numThreads>>>(m, n, A, x, y);

    cudaDeviceSynchronize();

    // Print first 3 and last 3 results (error check like SIMP spec requirement idk if still needed)
    printf("y[0..2] = { %f, %f, %f }\n", y[0], y[1], y[2]);
    printf("y[-3..-1] = { %f, %f, %f }\n", y[m-3], y[m-2], y[m-1]);

    //Floating-point tolerant error check
    float tol = 1e-3f;
    size_t err_count = 0;

    for (size_t i = 0; i < m; i++){
        float ref = 0.0f;
        for (size_t j = 0; j < n; j++){
            ref += A[i*n + j] * x[j];
        }
        if (fabsf(ref - y[i]) > tol)
            err_count++;
    }

    printf("Error count (CUDA program): %lu\n", (unsigned long)err_count);

    cudaFree(A);
    cudaFree(x);
    cudaFree(y);
    return 0;
}


Writing CUDA_MATVEC_VAR2.cu


In [9]:
%%bash
# nvcc CUDA_MATVEC_VAR2.cu -o CUDA_MATVEC_VAR2 -Wno-deprecated-gpu-targets
nvcc CUDA_MATVEC_VAR2.cu -o CUDA_MATVEC_VAR2 # for GPU (Tesla T4)

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [10]:
%%bash
nvprof ./CUDA_MATVEC_VAR2

==1008071== NVPROF is profiling process 1008071, command: ./CUDA_MATVEC_VAR2


*** function = MATVEC (float)
m = 4096, n = 4096 (A elements = 16777216)
numBlocks = 4, numThreads = 1024
y[0..2] = { -110.005165, -109.688553, -109.370766 }
y[-3..-1] = { 185.623337, 185.726227, 185.830673 }
Error count (CUDA program): 0


==1008071== Profiling application: ./CUDA_MATVEC_VAR2
==1008071== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:  100.00%  167.14ms        30  5.5713ms  3.6819ms  60.194ms  matvec(unsigned long, unsigned long, float const *, float const *, float*)
      API calls:   90.15%  1.63325s         3  544.42ms  27.580us  1.63248s  cudaMallocManaged
                    9.19%  166.55ms         1  166.55ms  166.55ms  166.55ms  cudaDeviceSynchronize
                    0.42%  7.5673ms         3  2.5224ms  445.64us  6.4637ms  cudaFree
                    0.18%  3.1794ms        30  105.98us  10.328us  2.4939ms  cudaLaunchKernel
                    0.04%  804.14us       114  7.0530us     140ns  449.16us  cuDeviceGetAttribute
                    0.01%  253.69us         1  253.69us  253.69us  253.69us  cuDeviceGetName
                    0.00%  28.269us         1  28.269us  28.269us  28.269us  cuDeviceTotalMem
                    0.0

In [11]:
%%bash
nsys profile -o CUDA_MATVEC_VAR2 ./CUDA_MATVEC_VAR2

         This may increase runtime overhead and the likelihood of false
         dependencies across CUDA Streams. If you wish to avoid this, please
         disable the feature with --cuda-event-trace=false.
Try the 'nsys status --environment' command to learn more.

Try the 'nsys status --environment' command to learn more.



*** function = MATVEC (float)
m = 4096, n = 4096 (A elements = 16777216)
numBlocks = 4, numThreads = 1024
y[0..2] = { -110.005165, -109.688553, -109.370766 }
y[-3..-1] = { 185.623337, 185.726227, 185.830673 }
Error count (CUDA program): 0
Generating '/tmp/nsys-report-395f.qdstrm'
[1/1] [========================100%] CUDA_MATVEC_VAR2.nsys-rep
Generated:
	/home/jupyter-aebrahm_ramos@dlsu-ac854/CUDA-MP/CUDA_MATVEC_VAR2.nsys-rep


---
---
# Variant 3: CUDA program version using a grid-stride loop with prefetching but without page creation and without mem advise
---

In [12]:
%%writefile CUDA_MATVEC_VAR3.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

// CUDA MATVEC kernel (grid-stride loop)
__global__
void matvec(size_t m, size_t n, const float *A, const float *x, float *y){
    int row  = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    for (int i = row; i < (int)m; i += stride){
        float sum = 0.0f;
        for (size_t j = 0; j < n; j++){
            sum += A[i*n + j] * x[j];
        }
        y[i] = sum;
    }
}

int main() {
    const size_t m = 4096;
    const size_t n = 4096;
    const size_t MATRIX_SIZE = m * n;
    const size_t MATRIX_BYTES = MATRIX_SIZE * sizeof(float);
    const size_t VECTOR_BYTES = n * sizeof(float);
    const size_t LOOPS = 30;

    float *A, *x, *y;
    cudaMallocManaged(&A, MATRIX_BYTES);
    cudaMallocManaged(&x, VECTOR_BYTES);
    cudaMallocManaged(&y, VECTOR_BYTES);

    //get gpu id
    int device = -1;
    cudaGetDevice(&device);

    // Initialize A and x
    for (size_t i = 0; i < m; i++){
        for (size_t j = 0; j < n; j++){
            A[i*n + j] = sinf(i * 0.002f + j * 0.001f);
        }
    }
    for (size_t i = 0; i < n; i++)
        x[i] = cosf(i * 0.003f);

    //Prefetch data from CPU-GPU
    cudaMemPrefetchAsync(A, MATRIX_BYTES, device, NULL);
    cudaMemPrefetchAsync(x, VECTOR_BYTES, device, NULL);

    // CUDA kernel launch setup
    size_t numThreads = 1024;
    size_t numBlocks = (m + numThreads - 1) / numThreads;

    printf("*** function = MATVEC (float)\n");
    printf("m = %lu, n = %lu (A elements = %lu)\n", m, n, MATRIX_SIZE);
    printf("numBlocks = %lu, numThreads = %lu\n",
           (unsigned long)numBlocks, (unsigned long)numThreads);

    // Multiple runs for nvprof timing
    for (size_t i = 0; i < LOOPS; i++)
        matvec<<<numBlocks, numThreads>>>(m, n, A, x, y);

    cudaDeviceSynchronize();

    //prefetch data from gpu-cpu
    cudaMemPrefetchAsync(y, VECTOR_BYTES, cudaCpuDeviceId);
    cudaMemPrefetchAsync(x, VECTOR_BYTES, cudaCpuDeviceId);
    cudaMemPrefetchAsync(A, MATRIX_BYTES, cudaCpuDeviceId);

    // Print first 3 and last 3 results (error check like SIMP spec requirement idk if still needed)
    printf("y[0..2] = { %f, %f, %f }\n", y[0], y[1], y[2]);
    printf("y[-3..-1] = { %f, %f, %f }\n", y[m-3], y[m-2], y[m-1]);

    //Floating-point tolerant error check
    float tol = 1e-3f;
    size_t err_count = 0;

    for (size_t i = 0; i < m; i++){
        float ref = 0.0f;
        for (size_t j = 0; j < n; j++){
            ref += A[i*n + j] * x[j];
        }
        if (fabsf(ref - y[i]) > tol)
            err_count++;
    }

    printf("Error count (CUDA program): %lu\n", (unsigned long)err_count);

    cudaFree(A);
    cudaFree(x);
    cudaFree(y);
    return 0;
}


Writing CUDA_MATVEC_VAR3.cu


In [13]:
%%bash
#nvcc CUDA_MATVEC_VAR3.cu -o CUDA_MATVEC_VAR3 -Wno-deprecated-gpu-targets
nvcc CUDA_MATVEC_VAR3.cu -o CUDA_MATVEC_VAR3 # for GPU (Tesla T4)

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [14]:
%%bash
nvprof ./CUDA_MATVEC_VAR3

==1008203== NVPROF is profiling process 1008203, command: ./CUDA_MATVEC_VAR3


*** function = MATVEC (float)
m = 4096, n = 4096 (A elements = 16777216)
numBlocks = 4, numThreads = 1024
y[0..2] = { -110.005165, -109.688553, -109.370766 }
y[-3..-1] = { 185.623337, 185.726227, 185.830673 }
Error count (CUDA program): 0


==1008203== Profiling application: ./CUDA_MATVEC_VAR3
==1008203== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:  100.00%  101.92ms        30  3.3975ms  3.3319ms  4.5632ms  matvec(unsigned long, unsigned long, float const *, float const *, float*)
      API calls:   90.46%  1.48924s         3  496.41ms  137.33us  1.48791s  cudaMallocManaged
                    6.16%  101.45ms         1  101.45ms  101.45ms  101.45ms  cudaDeviceSynchronize
                    2.63%  43.332ms         5  8.6663ms  330.94us  30.298ms  cudaMemPrefetchAsync
                    0.48%  7.9312ms         3  2.6437ms  594.12us  6.0599ms  cudaFree
                    0.21%  3.4051ms        30  113.50us  11.659us  2.8277ms  cudaLaunchKernel
                    0.04%  613.74us       114  5.3830us     124ns  231.10us  cuDeviceGetAttribute
                    0.01%  153.43us         1  153.43us  153.43us  153.43us  cuDeviceGetName
                   

In [15]:
%%bash
nsys profile -o CUDA_MATVEC_VAR3 ./CUDA_MATVEC_VAR3

         This may increase runtime overhead and the likelihood of false
         dependencies across CUDA Streams. If you wish to avoid this, please
         disable the feature with --cuda-event-trace=false.
Try the 'nsys status --environment' command to learn more.

Try the 'nsys status --environment' command to learn more.



*** function = MATVEC (float)
m = 4096, n = 4096 (A elements = 16777216)
numBlocks = 4, numThreads = 1024
y[0..2] = { -110.005165, -109.688553, -109.370766 }
y[-3..-1] = { 185.623337, 185.726227, 185.830673 }
Error count (CUDA program): 0
Generating '/tmp/nsys-report-4f0b.qdstrm'
[1/1] [========================100%] CUDA_MATVEC_VAR3.nsys-rep
Generated:
	/home/jupyter-aebrahm_ramos@dlsu-ac854/CUDA-MP/CUDA_MATVEC_VAR3.nsys-rep


---
---
# Variant 4: CUDA program version using a grid-stride loop with prefetch, with page creation but without mem advise
---

In [16]:
%%writefile CUDA_MATVEC_VAR4.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

// CUDA MATVEC kernel (grid-stride loop)
__global__
void matvec(size_t m, size_t n, const float *A, const float *x, float *y){
    int row  = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    for (int i = row; i < (int)m; i += stride){
        float sum = 0.0f;
        for (size_t j = 0; j < n; j++){
            sum += A[i*n + j] * x[j];
        }
        y[i] = sum;
    }
}

int main() {
    const size_t m = 4096;
    const size_t n = 4096;
    const size_t MATRIX_SIZE = m * n;
    const size_t MATRIX_BYTES = MATRIX_SIZE * sizeof(float);
    const size_t VECTOR_BYTES = n * sizeof(float);
    const size_t LOOPS = 30;

    float *A, *x, *y;
    cudaMallocManaged(&A, MATRIX_BYTES);
    cudaMallocManaged(&x, VECTOR_BYTES);
    cudaMallocManaged(&y, VECTOR_BYTES);

    int device = -1;
    cudaGetDevice(&device);

    //prefetch data to create cpu page memory
    cudaMemPrefetchAsync(A,MATRIX_BYTES,cudaCpuDeviceId,NULL);
    cudaMemPrefetchAsync(x,VECTOR_BYTES,cudaCpuDeviceId,NULL);

    //prefetch data to create gpu page memory
    cudaMemPrefetchAsync(y,MATRIX_BYTES,device,NULL);

    // Initialize A and x
    for (size_t i = 0; i < m; i++){
        for (size_t j = 0; j < n; j++){
            A[i*n + j] = sinf(i * 0.002f + j * 0.001f);
        }
    }
    for (size_t i = 0; i < n; i++)
        x[i] = cosf(i * 0.003f);

    //Prefetch data from cpu-gpu
    cudaMemPrefetchAsync(A,MATRIX_BYTES,device,NULL);
    cudaMemPrefetchAsync(x,VECTOR_BYTES,device,NULL);

    // CUDA kernel launch setup
    size_t numThreads = 1024;
    size_t numBlocks = (m + numThreads - 1) / numThreads;

    printf("*** function = MATVEC (float)\n");
    printf("m = %lu, n = %lu (A elements = %lu)\n", m, n, MATRIX_SIZE);
    printf("numBlocks = %lu, numThreads = %lu\n",
           (unsigned long)numBlocks, (unsigned long)numThreads);

    // Multiple runs for nvprof timing
    for (size_t i = 0; i < LOOPS; i++)
        matvec<<<numBlocks, numThreads>>>(m, n, A, x, y);

    cudaDeviceSynchronize();

    //prefetch data from gpu-cpu
    cudaMemPrefetchAsync(A,MATRIX_BYTES,cudaCpuDeviceId,NULL);
    cudaMemPrefetchAsync(A,VECTOR_BYTES,cudaCpuDeviceId,NULL);
    cudaMemPrefetchAsync(A,MATRIX_BYTES,cudaCpuDeviceId,NULL);

    // Print first 3 and last 3 results (error check like SIMP spec requirement idk if still needed)
    printf("y[0..2] = { %f, %f, %f }\n", y[0], y[1], y[2]);
    printf("y[-3..-1] = { %f, %f, %f }\n", y[m-3], y[m-2], y[m-1]);

    //Floating-point tolerant error check
    float tol = 1e-3f;
    size_t err_count = 0;

    for (size_t i = 0; i < m; i++){
        float ref = 0.0f;
        for (size_t j = 0; j < n; j++){
            ref += A[i*n + j] * x[j];
        }
        if (fabsf(ref - y[i]) > tol)
            err_count++;
    }

    printf("Error count (CUDA program): %lu\n", (unsigned long)err_count);

    cudaFree(A);
    cudaFree(x);
    cudaFree(y);
    return 0;
}


Writing CUDA_MATVEC_VAR4.cu


In [17]:
%%bash
#nvcc CUDA_MATVEC_VAR4.cu -o CUDA_MATVEC_VAR4 -Wno-deprecated-gpu-targets
nvcc CUDA_MATVEC_VAR4.cu -o CUDA_MATVEC_VAR4 # for GPU (Tesla T4)

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [18]:
%%bash
nvprof ./CUDA_MATVEC_VAR4

==1008335== NVPROF is profiling process 1008335, command: ./CUDA_MATVEC_VAR4


*** function = MATVEC (float)
m = 4096, n = 4096 (A elements = 16777216)
numBlocks = 4, numThreads = 1024
y[0..2] = { -110.005165, -109.688553, -109.370766 }
y[-3..-1] = { 185.623337, 185.726227, 185.830673 }
Error count (CUDA program): 0


==1008335== Profiling application: ./CUDA_MATVEC_VAR4
==1008335== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:  100.00%  224.16ms        30  7.4722ms  5.8564ms  8.5534ms  matvec(unsigned long, unsigned long, float const *, float const *, float*)
      API calls:   80.95%  1.66656s         3  555.52ms  160.38us  1.66538s  cudaMallocManaged
                   10.96%  225.60ms         1  225.60ms  225.60ms  225.60ms  cudaDeviceSynchronize
                    7.45%  153.30ms         8  19.162ms  174.82us  73.828ms  cudaMemPrefetchAsync
                    0.34%  7.0332ms         3  2.3444ms  418.43us  6.0173ms  cudaFree
                    0.25%  5.2468ms        30  174.89us  11.550us  4.6452ms  cudaLaunchKernel
                    0.03%  622.37us       114  5.4590us     115ns  230.05us  cuDeviceGetAttribute
                    0.01%  251.88us         1  251.88us  251.88us  251.88us  cuDeviceGetName
                   

In [19]:
%%bash
nsys profile -o CUDA_MATVEC_VAR4 ./CUDA_MATVEC_VAR4

         This may increase runtime overhead and the likelihood of false
         dependencies across CUDA Streams. If you wish to avoid this, please
         disable the feature with --cuda-event-trace=false.
Try the 'nsys status --environment' command to learn more.

Try the 'nsys status --environment' command to learn more.



*** function = MATVEC (float)
m = 4096, n = 4096 (A elements = 16777216)
numBlocks = 4, numThreads = 1024
y[0..2] = { -110.005165, -109.688553, -109.370766 }
y[-3..-1] = { 185.623337, 185.726227, 185.830673 }
Error count (CUDA program): 0
Generating '/tmp/nsys-report-b1a0.qdstrm'
[1/1] [========================100%] CUDA_MATVEC_VAR4.nsys-rep
Generated:
	/home/jupyter-aebrahm_ramos@dlsu-ac854/CUDA-MP/CUDA_MATVEC_VAR4.nsys-rep


---
---
# Variant 5: CUDA program version using a grid-stride loop with prefetch, with page creation and with mem advise
---

In [20]:
%%writefile CUDA_MATVEC_VAR5.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

// CUDA MATVEC kernel (grid-stride loop)
__global__
void matvec(size_t m, size_t n, const float *A, const float *x, float *y){
    int row  = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    for (int i = row; i < (int)m; i += stride){
        float sum = 0.0f;
        for (size_t j = 0; j < n; j++){
            sum += A[i*n + j] * x[j];
        }
        y[i] = sum;
    }
}

int main() {
    const size_t m = 4096;
    const size_t n = 4096;
    const size_t MATRIX_SIZE = m * n;
    const size_t MATRIX_BYTES = MATRIX_SIZE * sizeof(float);

    const size_t VECTOR_BYTES = n * sizeof(float);
    const size_t LOOPS = 30;

    float *A, *x, *y;
    cudaMallocManaged(&A, MATRIX_BYTES);
    cudaMallocManaged(&x, VECTOR_BYTES);
    cudaMallocManaged(&y, VECTOR_BYTES);

    int device = -1;
    cudaGetDevice(&device);

    //memory advise
    cudaMemAdvise(A, MATRIX_BYTES, cudaMemAdviseSetReadMostly, device);
    cudaMemAdvise(x, VECTOR_BYTES, cudaMemAdviseSetReadMostly, device);

    cudaMemAdvise(y, VECTOR_BYTES, cudaMemAdviseSetReadMostly, device);

    cudaMemAdvise(A, MATRIX_BYTES, cudaMemAdviseSetPreferredLocation, cudaCpuDeviceId);
    cudaMemAdvise(x, VECTOR_BYTES, cudaMemAdviseSetPreferredLocation, cudaCpuDeviceId);
    cudaMemAdvise(y, VECTOR_BYTES, cudaMemAdviseSetPreferredLocation, cudaCpuDeviceId);


    //prefetch data to create cpu page memory
    cudaMemPrefetchAsync(A,MATRIX_BYTES,cudaCpuDeviceId,NULL);
    cudaMemPrefetchAsync(x,VECTOR_BYTES,cudaCpuDeviceId,NULL);

    //prefetch data to create gpu page memory
    cudaMemPrefetchAsync(y,VECTOR_BYTES,device,NULL);

    //Initialize A and X
    for(size_t i = 0; i< m; i++) {
      for(size_t j = 0; j < n; j++) {
        A[i*n + j] = sinf(i * 0.002f + j * 0.001f);
      }
    }

    for (size_t i =0; i < n; i++)
      x[i] = cosf(i * 0.003f);

    //prefetch data from cpu to gpu
    cudaMemPrefetchAsync(A,MATRIX_BYTES,device,NULL);
    cudaMemPrefetchAsync(x,VECTOR_BYTES,device,NULL);

    // cuda kernel setup
    size_t numThreads = 1024;
    size_t numBlock = (m + numThreads - 1) / numThreads;

    printf("** function = MATVEC (float)\n");
    printf("m = %lu, n=%lu (A elements = %lu)\n", m, n, MATRIX_SIZE);
    printf("numBlocks = %lu, numThreads = %lu\n", numBlock, numThreads);

    // run multiple times for testing/nvprof timing
    for (size_t i = 0; i < LOOPS; i++)
      matvec<<<numBlock, numThreads>>>(m, n, A, x, y);

    cudaDeviceSynchronize();

    //prefetch data from gpu to cpu page memory
    cudaMemPrefetchAsync(A,MATRIX_BYTES,cudaCpuDeviceId,NULL);
    cudaMemPrefetchAsync(x,VECTOR_BYTES,cudaCpuDeviceId,NULL);
    cudaMemPrefetchAsync(y,VECTOR_BYTES,cudaCpuDeviceId,NULL);

    // Print first 3 and last 3 results
    printf("y[0..2] = { %f, %f, %f }\n", y[0], y[1], y[2]);
    printf("y[-3..-1] = { %f, %f, %f }\n", y[m-3], y[m-2], y[m-1]);

    float tol = 1e-3f;
    size_t err_count = 0;

    for(size_t i = 0; i < m; i++) {
      float ref = 0.0f;
      for (size_t j = 0; j < n; j++){
        ref += A[i*n + j] * x[j];
      }
      if (fabs(ref - y[i]) > tol)
        err_count++;
    }

    printf("Error count (CUDA program): %lu\n",(unsigned long)err_count);

    cudaFree(A);
    cudaFree(x);
    cudaFree(y);
    return 0;
}

Writing CUDA_MATVEC_VAR5.cu


In [21]:
%%bash
#nvcc CUDA_MATVEC_VAR5.cu -o CUDA_MATVEC_VAR5 -Wno-deprecated-gpu-targets
nvcc CUDA_MATVEC_VAR5.cu -o CUDA_MATVEC_VAR5 # for GPU (Tesla T4)

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [22]:
%%bash
nvprof ./CUDA_MATVEC_VAR5

==1008465== NVPROF is profiling process 1008465, command: ./CUDA_MATVEC_VAR5


** function = MATVEC (float)
m = 4096, n=4096 (A elements = 16777216)
numBlocks = 4, numThreads = 1024
y[0..2] = { -110.005165, -109.688553, -109.370766 }
y[-3..-1] = { 185.623337, 185.726227, 185.830673 }
Error count (CUDA program): 0


==1008465== Profiling application: ./CUDA_MATVEC_VAR5
==1008465== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:  100.00%  99.167ms        30  3.3056ms  3.2993ms  3.3117ms  matvec(unsigned long, unsigned long, float const *, float const *, float*)
      API calls:   90.16%  1.83378s         3  611.26ms  24.668us  1.83326s  cudaMallocManaged
                    4.86%  98.791ms         1  98.791ms  98.791ms  98.791ms  cudaDeviceSynchronize
                    4.23%  86.069ms         8  10.759ms  307.28us  61.434ms  cudaMemPrefetchAsync
                    0.54%  11.038ms         3  3.6792ms  585.57us  9.0597ms  cudaFree
                    0.15%  3.1482ms        30  104.94us  8.9210us  2.6851ms  cudaLaunchKernel
                    0.03%  551.95us       114  4.8410us      98ns  268.61us  cuDeviceGetAttribute
                    0.02%  331.11us         6  55.185us  10.571us  214.39us  cudaMemAdvise
                    0

In [23]:
%%bash
nsys profile -o CUDA_MATVEC_VAR5 ./CUDA_MATVEC_VAR5

         This may increase runtime overhead and the likelihood of false
         dependencies across CUDA Streams. If you wish to avoid this, please
         disable the feature with --cuda-event-trace=false.
Try the 'nsys status --environment' command to learn more.

Try the 'nsys status --environment' command to learn more.



** function = MATVEC (float)
m = 4096, n=4096 (A elements = 16777216)
numBlocks = 4, numThreads = 1024
y[0..2] = { -110.005165, -109.688553, -109.370766 }
y[-3..-1] = { 185.623337, 185.726227, 185.830673 }
Error count (CUDA program): 0
Generating '/tmp/nsys-report-71b0.qdstrm'
[1/1] [========================100%] CUDA_MATVEC_VAR5.nsys-rep
Generated:
	/home/jupyter-aebrahm_ramos@dlsu-ac854/CUDA-MP/CUDA_MATVEC_VAR5.nsys-rep


---
---
# Variant 6: Classic MemCopy method (no Unified memory)
---

In [9]:
%%writefile CUDA_MATVEC_VAR6.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

// CUDA MATVEC kernel (grid-stride loop)
__global__
void matvec(size_t m, size_t n, const float *A, const float *x, float *y){
    int row = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    for (int i = row; i < (int)m; i += stride){
        float sum = 0.0f;
        for (size_t j = 0; j < n; j++){
            sum += A[i*n + j] * x[j];
        }
        y[i] = sum;
    }
}

int main() {
    const size_t m = 4096;
    const size_t n = 4096;
    const size_t MATRIX_SIZE = m * n;
    const size_t MATRIX_BYTES = MATRIX_SIZE * sizeof(float);
    const size_t X_BYTES = n * sizeof(float);
    const size_t Y_BYTES = m * sizeof(float);
    const size_t LOOPS = 30;

    // Host memory allocation
    float *h_A = (float*)malloc(MATRIX_BYTES);
    float *h_x = (float*)malloc(X_BYTES);
    float *h_y = (float*)malloc(Y_BYTES);

    // Initialize A and x on host
    for (size_t i = 0; i < m; i++)
        for (size_t j = 0; j < n; j++)
            h_A[i*n + j] = sinf(i * 0.002f + j * 0.001f);

    for (size_t i = 0; i < n; i++)
        h_x[i] = cosf(i * 0.003f);

    for (size_t i = 0; i < m; i++)
        h_y[i] = 0.0f;

    // Device memory allocation
    float *d_A, *d_x, *d_y;
    cudaMalloc(&d_A, MATRIX_BYTES);
    cudaMalloc(&d_x, X_BYTES);
    cudaMalloc(&d_y, Y_BYTES);

    // Copy data from host to device
    cudaMemcpy(d_A, h_A, MATRIX_BYTES, cudaMemcpyHostToDevice);
    cudaMemcpy(d_x, h_x, X_BYTES, cudaMemcpyHostToDevice);
    cudaMemset(d_y, 0, Y_BYTES);

    // CUDA kernel launch setup
    size_t numThreads = 1024;
    size_t numBlocks = (m + numThreads - 1) / numThreads;

    printf("*** VARIANT 6: MATVEC with Classic MemCopy (No Unified Memory)\n");
    printf("m = %lu, n = %lu (A elements = %lu)\n", m, n, MATRIX_SIZE);
    printf("numBlocks = %lu, numThreads = %lu\n",
           (unsigned long)numBlocks, (unsigned long)numThreads);

    for (size_t i = 0; i < LOOPS; i++)
        matvec<<<numBlocks, numThreads>>>(m, n, d_A, d_x, d_y);

    cudaDeviceSynchronize();

    cudaMemcpy(h_y, d_y, Y_BYTES, cudaMemcpyDeviceToHost);

    printf("y[0] = %.8e, y[%lu] = %.8e, y[%lu] = %.8e\n",
       h_y[0], m-1, h_y[m-1], m/2, h_y[m/2]);


    size_t err_count = 0;
    float max_rel_err = 0.0f;
    for (size_t i = 0; i < m; i++){
        float ref = 0.0f;
        for (size_t j = 0; j < n; j++)
            ref += h_A[i*n + j] * h_x[j];

        // Use relative error for better comparison
        float abs_err = fabsf(ref - h_y[i]);
        float rel_err = abs_err / fmaxf(1.0f, fabsf(ref));

        if (rel_err > 1e-4f)  // 0.01% relative tolerance
            err_count++;

        max_rel_err = fmaxf(max_rel_err, rel_err);
}

printf("Error count (Variant 5): %lu\n", (unsigned long)err_count);
printf("Max relative error: %.6e\n", max_rel_err);

    cudaFree(d_A);
    cudaFree(d_x);
    cudaFree(d_y);
    free(h_A);
    free(h_x);
    free(h_y);

    return 0;
}

Overwriting CUDA_MATVEC_VAR6.cu


In [10]:
%%bash
# nvcc -O3 CUDA_MATVEC_VAR6.cu -o CUDA_MATVEC_VAR6
nvcc CUDA_MATVEC_VAR6.cu -o CUDA_MATVEC_VAR6 # for GPU (Tesla T4)

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [11]:
%%bash
nvprof ./CUDA_MATVEC_VAR6

==1009642== NVPROF is profiling process 1009642, command: ./CUDA_MATVEC_VAR6


*** VARIANT 6: MATVEC with Classic MemCopy (No Unified Memory)
m = 4096, n = 4096 (A elements = 16777216)
numBlocks = 4, numThreads = 1024
y[0] = -1.10005165e+02, y[4095] = 1.85830673e+02, y[2048] = -6.56716614e+01
Error count (Variant 5): 0
Max relative error: 7.226525e-05


==1009642== Profiling application: ./CUDA_MATVEC_VAR6
==1009642== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   56.75%  110.46ms        30  3.6821ms  3.4999ms  3.6942ms  matvec(unsigned long, unsigned long, float const *, float const *, float*)
                   43.24%  84.164ms         2  42.082ms  2.6560us  84.161ms  [CUDA memcpy HtoD]
                    0.00%  2.6560us         1  2.6560us  2.6560us  2.6560us  [CUDA memcpy DtoH]
                    0.00%  2.4640us         1  2.4640us  2.4640us  2.4640us  [CUDA memset]
      API calls:   85.61%  1.19453s         3  398.18ms  42.104us  1.19386s  cudaMalloc
                    7.89%  110.10ms         1  110.10ms  110.10ms  110.10ms  cudaDeviceSynchronize
                    6.17%  86.149ms         3  28.716ms  215.25us  85.410ms  cudaMemcpy
                    0.13%  1.8564ms         3  618.81us  122.64us  1.1609ms  cudaFree
                    0.12%  1.7043ms   

In [12]:
%%bash
nsys profile -o CUDA_MATVEC_VAR6 ./CUDA_MATVEC_VAR6

         This may increase runtime overhead and the likelihood of false
         dependencies across CUDA Streams. If you wish to avoid this, please
         disable the feature with --cuda-event-trace=false.
Try the 'nsys status --environment' command to learn more.

Try the 'nsys status --environment' command to learn more.



*** VARIANT 6: MATVEC with Classic MemCopy (No Unified Memory)
m = 4096, n = 4096 (A elements = 16777216)
numBlocks = 4, numThreads = 1024
y[0] = -1.10005165e+02, y[4095] = 1.85830673e+02, y[2048] = -6.56716614e+01
Error count (Variant 5): 0
Max relative error: 7.226525e-05


Failed to create '/home/jupyter-aebrahm_ramos@dlsu-ac854/CUDA-MP/CUDA_MATVEC_VAR6.nsys-rep': File exists.
Use `--force-overwrite true` to overwrite existing files.


Generating '/tmp/nsys-report-3f68.qdstrm'
[1/1] [========================100%] nsys-report-eacb.nsys-rep
Generated:
	/tmp/nsys-report-eacb.nsys-rep
